In [1]:
import sys
sys.path.append("../")

In [2]:
import torch
from qmpsqsc.models import mpsqsc
from qmpsqsc.models import qmps
from importlib import reload

reload(mpsqsc)

<module 'qmpsqsc.models.mpsqsc' from '/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/__init__.py'>

In [95]:
import torch.nn.functional as F

L = 8
chi = 2
d = 2
dtype = torch.float64
qscr = mpsqsc.MpsQsc(L, chi, d, dtype=dtype, init="stacked")
allup = torch.zeros(L, d, dtype = dtype)
allup[:, 0] = 1.0
alldown = torch.zeros(L, d, dtype = dtype)
alldown[:, 1] = 1.0

mps_allup = mpsqsc.build_product_state(L, d, allup)
mps_alldown = mpsqsc.build_product_state(L, d, alldown)

mps_allup = mps_allup.normalize()
mps_alldown = mps_alldown.normalize()

ghz = mpsqsc.build_ghz_state(L, d, chi, dtype = torch.complex128)
ghz = ghz.normalize()

# qscr = qscr.canonicalize(truncate=True, normalize=True)

In [4]:
ghz = mpsqsc.build_ghz_state(L, d, chi)
ghz2 = mpsqsc.build_ghz_state(L, d, chi)
# Construct |000> - |111>
As = ghz2.As
As[0][:, 1] = -As[0][:, 1]
ghz2.set_As(As)
qsc = mpsqsc.build_2qsc_from_mpstate(ghz, ghz2)
# qsc = qsc.truncate_bond_dimension(2)

# This classifier return 0 for ghz state and 50 / 50 for all up or all down

In [5]:
import random
from typing import List, Tuple, Dict
import torch

@torch.no_grad()
def _ensure_same_device_dtype(state, device, dtype):
    if getattr(state, "device", None) != device or getattr(state, "dtype", None) != dtype:
        state.to(device=device, dtype=dtype)
    return state

def _amps_to_probs_batch(amps: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """
    Convert a batch of amplitudes to Born probabilities.
      amps: (B, 2) real or complex
      returns probs: (B, 2), each row sums to 1
    """
    if torch.is_complex(amps):
        abs_sq = (amps.conj() * amps).real
    else:
        abs_sq = amps * amps
    denom = abs_sq.sum(dim=-1, keepdim=True).clamp_min(eps)
    return abs_sq / denom

def train_classifier_qsc_on_ghz_vs_rho(
    qsc,
    mpsghz,
    mps_allup,
    mps_alldown,
    *,
    steps: int = 2000,
    lr: float = 1e-2,
    batch_size: int = 16,          # kept for API compatibility but ignored
    ghz_fraction: float = 0.5,     # ignored (fixed composition batch)
    weight_decay: float = 0.0,
    grad_clip: float | None = None,
    seed: int | None = 0,
    log_every: int = 100,
) -> Dict[str, List[float]]:
    """
    Train qsc so that it outputs class 1 for GHZ, class 0 for the mixture ρ.

    CHANGE: No softmax. We treat the 2-D output as amplitudes a,
            compute probabilities p_k = |a_k|^2 / sum_j |a_j|^2,
            and minimize NLL:  -log p_y.

    Each step uses a fixed batch of 4 samples:
        [GHZ, GHZ, all-up, all-down] with labels [1, 1, 0, 0].
    """
    # if seed is not None:
    #     torch.manual_seed(seed)
    #     random.seed(seed)

    device, dtype = qsc.device, qsc.dtype

    # Ensure states match qsc's device/dtype
    _ensure_same_device_dtype(mpsghz, device, dtype)
    _ensure_same_device_dtype(mps_allup, device, dtype)
    _ensure_same_device_dtype(mps_alldown, device, dtype)

    # Collect parameters (works for nn.Module or your minimal class exposing .parameters())
    params = qsc.As
    optim = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)

    history: Dict[str, List[float]] = {"loss": [], "acc": []}


    fixed_labels = torch.tensor([0, 0, 1, 1], dtype=torch.long, device=device)
    X = torch.tensor([[0, 1], [1, 0]], device=device, dtype=dtype)

    for step in range(1, steps + 1):
        # Fixed batch content per step
        fixed_states = [mpsghz.copy(), mpsghz.copy(), mps_allup.copy(), mps_alldown.copy()]

        # Apply random flip to random sites

        for i in range(len(fixed_states)):
            r = torch.rand(1).item()
            # print(r)
            if r < 0:
                ind = random.randint(0, L - 1)
                # print("flipped")
                A = fixed_states[i].As[ind]
                if ind != 0 and ind != L - 1:
                    A = torch.einsum("iaj, ab -> ibj", A, X)
                elif ind == 0:
                    A = torch.einsum("aj, ab -> bj", A, X)
                elif ind == L - 1:
                    A = torch.einsum("ia, ab -> ib", A, X)
                fixed_states[i].As[ind] = A
        if hasattr(qsc, "train"):
            qsc.train()

        optim.zero_grad(set_to_none=True)

        # Optionally shuffle the 4 examples each step
        idx = [0, 1, 2, 3]
        states = [fixed_states[i] for i in idx]
        y = fixed_labels[idx]                       # shape (4,)

        # ---- forward: amplitudes -> Born probs ----
        amps_list = [qsc.contract_with_state(st) for st in states]   # each (2,)
        amps_batch = torch.stack(amps_list, dim=0)                   # (4, 2)
        probs = _amps_to_probs_batch(amps_batch)                     # (4, 2)
        # print(probs)

        # ---- loss: negative log-likelihood on target class ----
        eps = 1e-12
        p_y = probs.gather(1, y.view(-1, 1)).squeeze(1)              # (4,)
        nll = -torch.log(p_y.clamp_min(eps)).mean()
        loss = nll

        # ---- backward/step ----
        loss.backward()
        if grad_clip is not None and len(params) > 0:
            torch.nn.utils.clip_grad_norm_(params, max_norm=grad_clip)
        optim.step()

        # ---- metric: accuracy by argmax over Born probs ----
        with torch.no_grad():
            pred = probs.argmax(dim=-1)                              # (4,)
            acc = (pred == y).float().mean().item()

        history["loss"].append(float(loss.item()))
        history["acc"].append(acc)

        if (log_every is not None) and (step % log_every == 0):
            print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc:.3f}")

    return history


In [6]:
# You already created these:
# L = 10; chi = 2; d = 2
# qsc = mps_classifier.MpsQsc(L, chi, d)
# mps_allup, mps_alldown, mpsghz prepared and normalized
# qsc = mps_classifier.build_qsc_from_mpstate(mpsghz)   # if you prefer that init

hist = train_classifier_qsc_on_ghz_vs_rho(
    qsc,
    mpsghz=ghz,
    mps_allup=mps_allup,
    mps_alldown=mps_alldown,
    steps=100,
    lr=1e-2,
    weight_decay=0.0,
    grad_clip=1.0,   # optional
    seed=1209387,
    log_every=1,
)

[step     1] loss=0.346574  acc=0.500
[step     2] loss=0.052013  acc=1.000
[step     3] loss=0.008611  acc=1.000
[step     4] loss=0.000001  acc=1.000
[step     5] loss=0.002420  acc=1.000
[step     6] loss=0.005037  acc=1.000
[step     7] loss=0.005890  acc=1.000
[step     8] loss=0.005329  acc=1.000
[step     9] loss=0.004071  acc=1.000
[step    10] loss=0.002679  acc=1.000
[step    11] loss=0.001486  acc=1.000
[step    12] loss=0.000638  acc=1.000
[step    13] loss=0.000160  acc=1.000
[step    14] loss=0.000002  acc=1.000
[step    15] loss=0.000076  acc=1.000
[step    16] loss=0.000287  acc=1.000
[step    17] loss=0.000546  acc=1.000
[step    18] loss=0.000783  acc=1.000
[step    19] loss=0.000951  acc=1.000
[step    20] loss=0.001027  acc=1.000
[step    21] loss=0.001011  acc=1.000
[step    22] loss=0.000916  acc=1.000
[step    23] loss=0.000766  acc=1.000
[step    24] loss=0.000588  acc=1.000
[step    25] loss=0.000409  acc=1.000
[step    26] loss=0.000250  acc=1.000
[step    27]

KeyboardInterrupt: 

In [21]:
ghz3 = mpsqsc.build_ghz_state(L, d, chi)
ghz3.As[0][:] = torch.tensor([[0, 1], [1, 0]])
ghz3 = ghz3.normalize()

qsc.contract_with_state(ghz3)

tensor([0., 0.], dtype=torch.float64, grad_fn=<ViewBackward0>)

In [22]:
qsc_chi2 = qsc.truncate_bond_dimension(2)
qsc_chi2.contract_with_state(ghz), qsc_chi2.contract_with_state(mps_allup), qsc_chi2.contract_with_state(mps_alldown)

(tensor([-53.5447, -20.9102], dtype=torch.float64, grad_fn=<ViewBackward0>),
 tensor([-24.8281,  99.7386], dtype=torch.float64, grad_fn=<ViewBackward0>),
 tensor([ -28.7166, -120.6488], dtype=torch.float64, grad_fn=<ViewBackward0>))

In [23]:
hist = train_classifier_qsc_on_ghz_vs_rho(
    qsc_chi2,
    mpsghz=ghz,
    mps_allup=mps_allup,
    mps_alldown=mps_alldown,
    steps=20,
    lr=1e-4,
    weight_decay=0.0,
    grad_clip=None,
    seed=0,
    log_every=1,
)

[step     1] loss=0.099776  acc=1.000
[step     2] loss=0.086593  acc=1.000
[step     3] loss=0.074571  acc=1.000
[step     4] loss=0.063829  acc=1.000
[step     5] loss=0.054470  acc=1.000
[step     6] loss=0.046574  acc=1.000
[step     7] loss=0.040187  acc=1.000
[step     8] loss=0.035310  acc=1.000
[step     9] loss=0.031889  acc=1.000
[step    10] loss=0.029811  acc=1.000
[step    11] loss=0.028898  acc=1.000
[step    12] loss=0.028922  acc=1.000
[step    13] loss=0.029619  acc=1.000
[step    14] loss=0.030714  acc=1.000
[step    15] loss=0.031950  acc=1.000
[step    16] loss=0.033112  acc=1.000
[step    17] loss=0.034041  acc=1.000
[step    18] loss=0.034640  acc=1.000
[step    19] loss=0.034872  acc=1.000
[step    20] loss=0.034748  acc=1.000


In [115]:
qsc_chi2.save_to("data/qsc_chi2.pt")
# qsc_chi2 = mpsqsc.MpsQsc.load_model("data/qsc_chi2.pt")
# qsc_chi2 = mpsqsc.MpsQsc(L, 2, d, dtype=torch.float64)

In [116]:
from qmpsqsc.models.qmps import qMPS, construct_unitary_from_As
As = qsc.As
for i in range(L):
    As[i].data[:] += torch.randn_like(As[i]) * 1e-4
qsc_chi2 = qsc_chi2.canonicalize(truncate=True, normalize=True)
As = qsc_chi2.As

# add a random noise to As
Us, last = construct_unitary_from_As(As)
chi = 4
qmps = qMPS(L, chi, d, Us=Us, last_unitary=last)

In [117]:
rho = qmps._contract_circuit_with_state_ae(ghz)
pred = qsc_chi2.contract_with_state(ghz)

# normalize pred 
pred /= torch.linalg.norm(pred)

scale = rho.trace()
ghz.normalize(inplace=True, current_norm=torch.sqrt(scale))

rho = qmps._contract_circuit_with_state_ae(ghz)

# take diagonal part of rho
rho_diag = torch.diag(rho)
print("comfirm output from mps-qsc and qmps-qsc are the same", rho_diag - pred**2)

comfirm output from mps-qsc and qmps-qsc are the same tensor([-1.1102e-16,  1.1102e-16], dtype=torch.float64, grad_fn=<SubBackward0>)


/Users/keisuke/Documents/projects/mps4qsc/notebooks/../qmpsqsc/models/mpsqsc/mpsbase.py:472: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  nrm = self.norm(squared=False) if current_norm is None else torch.tensor(current_norm, device=self.device, dtype=self.dtype).detach().clone()


# Train qmps-qsc

In [118]:
import random
from typing import Dict, List, Tuple
import torch
from qmpsqsc.models.qmps.optimizer import StiefelAdam


# --- small helper to get parameters from qMPS ---------------------------

def _qmps_params(qmps):
    return [U.weight for U in qmps.U4] + [qmps.last_unitary.weight]


# 1) function to create a dataset ---------------------------------------

def create_ghz_rho_dataset(
    mps_ghz,
    mps_allup,
    mps_alldown,
    device, 
    dtype
) -> Tuple[List[torch.Tensor], torch.Tensor]:
    """
    Create a tiny dataset:
        GHZ, GHZ, all-up, all-down
    with labels:
        0, 0, 1, 1   (ρ=0, GHZ=1 or vice versa as you like)

    States are moved to qmps device/dtype and normalized using qmps.predict.
    """
    mps_ghz     = mps_ghz.to(device=device, dtype=dtype)
    mps_allup   = mps_allup.to(device=device, dtype=dtype)
    mps_alldown = mps_alldown.to(device=device, dtype=dtype)

    states = [mps_ghz, mps_ghz, mps_allup, mps_alldown]
    labels = torch.tensor([0,0,1,1], dtype=torch.long, device=device)

    # normalize the states once using qmps.predict (normalize=False)
    for s in states:
        pred = qmps.predict(s, normalize=False)  # probs up to scale
        scale = torch.sum(pred)
        s.normalize(inplace=True, current_norm=torch.sqrt(scale))

    return states, labels


# 2) function to calculate loss based on dataset and labels --------------

def qmps_ghz_rho_nll(
    qmps : qMPS,
    states: List[torch.Tensor],
    labels: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Compute NLL loss for the GHZ vs ρ dataset.

    Returns:
        loss: scalar tensor
        probs: (B, 2) tensor of class probabilities
    """
    # probs: (B, 2)
    probs = torch.stack([qmps.predict(s) for s in states], dim=0)

    # NLL loss: -log p_y
    p_y = probs.gather(1, labels.view(-1, 1)).squeeze(1)  # (B,)
    loss = -torch.log(p_y.clamp_min(1e-12)).mean()
    return loss, probs


# 3) training loop using the above two helpers ---------------------------

def train_qmps_classifier_ghz_vs_rho(
    qmps,               # type: qMPS
    mps_ghz,
    mps_allup,
    mps_alldown,
    steps: int = 2000,
    lr: float = 1e-3,
    seed: int | None = 0,
    log_every: int = 100,
) -> Dict[str, List[float]]:

    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)

    params = _qmps_params(qmps)
    optim = StiefelAdam(params, lr=lr)

    fixed_states, fixed_labels = create_ghz_rho_dataset(
        mps_ghz, mps_allup, mps_alldown,
        qmps.device, qmps.dtype
    )

    history: Dict[str, List[float]] = {"loss": [], "acc": []}

    for step in range(1, steps + 1):
        optim.zero_grad(set_to_none=True)

        # Shuffle the 4 examples each step
        idx = [0, 1, 2, 3]
        random.shuffle(idx)
        states = [fixed_states[i] for i in idx]
        y = fixed_labels[idx]

        loss, probs = qmps_ghz_rho_nll(qmps, states, y)
        loss.backward()
        optim.step()

        # Accuracy
        with torch.no_grad():
            pred = probs.argmax(dim=-1)
            acc = (pred == y).float().mean().item()

        history["loss"].append(float(loss.item()))
        history["acc"].append(acc)

        if log_every is not None and step % log_every == 0:
            print(f"[step {step:5d}] loss={loss.item():.6f}  acc={acc:.3f}")

    return history


In [119]:
w = 1
qmps.set_weights(w)

states, labels = create_ghz_rho_dataset(
    ghz,
    mps_allup,
    mps_alldown,
    qmps.device, qmps.dtype
)

qmps_ghz_rho_nll(qmps, states, labels)

(tensor(0.7375, dtype=torch.float64, grad_fn=<NegBackward0>),
 tensor([[0.3540, 0.6460],
         [0.3540, 0.6460],
         [0.3459, 0.6541],
         [0.3615, 0.6385]], dtype=torch.float64, grad_fn=<StackBackward0>))

In [120]:
ghz2 = ghz2.normalize()
mps_allup = mps_allup.normalize()
qmps._contract_circuit_with_state_ae(ghz3)

ValueError: Number of einsum subscripts, 40, must be equal to the number of operands, 224.

In [114]:
hist = train_qmps_classifier_ghz_vs_rho(
    qmps,
    ghz,
    mps_allup,
    mps_alldown,
    steps=1000,
    lr=1e-3 * 5,
    seed=0,
    log_every=1,
)

[step     1] loss=0.737476  acc=0.500
[step     2] loss=0.730119  acc=0.500
[step     3] loss=0.723374  acc=0.500
[step     4] loss=0.717270  acc=0.500
[step     5] loss=0.711837  acc=0.500
[step     6] loss=0.707100  acc=0.500
[step     7] loss=0.703076  acc=0.500
[step     8] loss=0.699767  acc=0.500
[step     9] loss=0.697160  acc=0.500
[step    10] loss=0.695230  acc=0.500
[step    11] loss=0.693929  acc=0.500
[step    12] loss=0.693195  acc=0.500
[step    13] loss=0.692941  acc=0.250
[step    14] loss=0.693066  acc=0.500
[step    15] loss=0.693456  acc=0.500
[step    16] loss=0.693993  acc=0.500
[step    17] loss=0.694565  acc=0.500
[step    18] loss=0.695080  acc=0.500
[step    19] loss=0.695469  acc=0.500
[step    20] loss=0.695694  acc=0.500
[step    21] loss=0.695741  acc=0.500
[step    22] loss=0.695622  acc=0.500
[step    23] loss=0.695363  acc=0.500
[step    24] loss=0.695001  acc=0.500
[step    25] loss=0.694574  acc=0.500
[step    26] loss=0.694118  acc=0.500
[step    27]

In [93]:
ghz3 = mpsqsc.build_ghz_state(L, d, 2)
ghz3.As[10][:] = torch.tensor([[0, 1], [1, 0]])
ghz3 = ghz3.normalize()

In [208]:
qmps.set_weights(1)

In [94]:
qmps.predict(ghz3, normalize=False)

tensor([0.5043, 0.4957], dtype=torch.float64, grad_fn=<DiagonalBackward0_copy>)